# Lab 21: How to call any model through one interface with LiteLLM

**Notebook 2 of 2: LiteLLM at the Chinook desk**

## Goal

Put LiteLLM between a working agent and its model. The agent is the desk agent of the Chinook music shop: a
support rep asks about a customer, and the agent looks the facts up in the shop's database. We build it on the
plain OpenAI client, and then point it at a LiteLLM gateway by changing two things, the client and the model's
name. Then we point the model's address at a place where nothing answers. An agent wired straight to that address stops with an error. The
agent behind the gateway still gives the rep the customer's three facts, and the gateway's reply headers say which
deployment answered each request, what it cost, and why the answer took longer.

## What you will learn

- How to build a small desk agent: three database lookups, their descriptions for the model, and a loop.
- How to write a gateway's config file with a primary deployment, a backup and a fallback, and start the gateway
  from the notebook.
- How to point an agent at the gateway by changing its client and its model name, and nothing inside the agent.
- How to read from the gateway's reply headers which deployment answered each request and what it cost.
- What a fallback saves when the primary cannot be reached, and what it costs in waiting time.

## From the playground to the desk

In notebook 1 we met LiteLLM while naming a kitten. Four ideas carry over:

- **One call for every vendor.** LiteLLM takes a call in the shape of OpenAI's Chat Completions and passes it on
  to the vendor named in front of the model: `openai/<model>`, `gemini/<model>`.
- **A deployment** is one entry in a list: a name of our own (`model_name`), what LiteLLM really calls for that
  name (`litellm_params`), and a label we choose (`model_info.id`).
- **A fallback** is a second name to try when a call to the first one fails.
- **The gateway** is a server that holds the deployments, the fallbacks and the vendor's key. Programs call it
  the way they would call OpenAI.

Now we take it seriously. A rep repeats the agent's answer to a customer on the phone. So when the model's
address does not answer, the rep still needs an answer, not an error. This notebook restates everything it uses, so it runs on
its own.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model and the prices from
`models.env`, and takes the key from `.env` without printing it.

In [1]:
from pathlib import Path
import json
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


---

## Part A: Build the desk agent on the plain OpenAI client

The shop's data is `data/chinook.db`, a SQLite file with the customers, their invoices and the support reps who
look after them. A rep at the desk asks things like:

> How many invoices does Helena Holý have, what do they add up to, and who is her support rep?

The question asks for three facts, and they sit in three tables. We write one small function for each lookup and
call them ourselves first, so that we know the right answers before any model is involved.

### The three lookups

- `find_customer(name)` finds a customer by full name and returns their `customer_id`.
- `get_invoices(customer_id)` counts that customer's invoices and adds up their total.
- `get_support_rep(customer_id)` names the rep who looks after them.

`mode=ro` in the first lines opens the database **read-only**, so no lookup can change the shop's data.

In [2]:
DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id."""
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}

In [3]:
customer = find_customer("Helena Holý")[0]

print(customer)
print(get_invoices(customer["customer_id"]))
print(get_support_rep(customer["customer_id"]))

{'customer_id': 6, 'name': 'Helena Holý', 'country': 'Czech Republic'}
{'customer_id': 6, 'invoice_count': 7, 'total': 49.62}
{'customer_id': 6, 'support_rep': 'Steve Johnson'}


These are the facts every answer in this notebook is checked against:

| Customer | `customer_id` | Invoices | Total | Support rep |
|---|---|---|---|---|
| Helena Holý | 6 | 7 | \$49.62 | Steve Johnson |

**Notice** the order the lookups need. `get_invoices` and `get_support_rep` take a `customer_id`, and only
`find_customer` can supply it.

### Describe the lookups to the model

The model never sees our Python. For each function we send a description, a **tool schema**, in the form Chat
Completions expects. The first one in full:

- `name`: what the model writes to ask for this tool.
- `description`: what the tool is for. The model decides from this text whether to ask for it.
- `parameters`: the arguments, as a JSON Schema. Here one string called `name`.
- `strict: True`: the model's arguments must match the schema exactly.

In [4]:
find_customer_tool = {
    "type": "function",
    "function": {
        "name": "find_customer",
        "description": "Find a shop customer by full name. Returns a list of matches with their customer_id.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "First and last name, for example 'Ana Silva'"}},
            "required": ["name"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

The other two have the same shape. Each takes one whole number, `customer_id`. `TOOLS` is the list we send to the
model. `FUNCTIONS` maps each tool's name to its Python function, so that our code can run whichever one the model
asks for.

In [5]:
get_invoices_tool = {
    "type": "function",
    "function": {
        "name": "get_invoices",
        "description": "Count a customer's invoices and add up their total in US dollars.",
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "integer"}},
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

get_support_rep_tool = {
    "type": "function",
    "function": {
        "name": "get_support_rep",
        "description": "Name the support rep who looks after a customer.",
        "parameters": {
            "type": "object",
            "properties": {"customer_id": {"type": "integer"}},
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

TOOLS = [find_customer_tool, get_invoices_tool, get_support_rep_tool]
FUNCTIONS = {"find_customer": find_customer, "get_invoices": get_invoices, "get_support_rep": get_support_rep}

### The agent: a loop around one model call

A model asked a question with tools does one of two things: it answers, or it asks for one or more tools. An
agent is the loop that keeps going until it answers. `ask_desk` is that loop, and it takes three arguments:

- `client`: the OpenAI client to call the model with;
- `model`: the model's name;
- `question`: the rep's question.

Keep an eye on the first two. They are the only things we change for the rest of this notebook.

What happens on each pass, step by step:

1. Call the model with the conversation so far and the three tool schemas.
2. If the reply has no tool call, it is the answer: return its text.
3. Otherwise add the model's request to the conversation, and run each lookup it asked for: `json.loads` turns the
   arguments string into a dictionary, and `FUNCTIONS[...](**arguments)` calls the function of that name with it.
4. Add each result to the conversation as a message with the role `tool`, carrying the id of the call it answers,
   and go round again.

The `print` in step 3 shows every lookup as it runs.

In [6]:
from openai import OpenAI, omit

INSTRUCTIONS = (
    "You help the support reps of a music shop. Look up every fact with the tools; "
    "never answer from memory. Answer in one or two sentences."
)


def ask_desk(client, model, question):
    """Answer a rep's question: call the model, run the lookups it asks for, repeat until it answers."""
    messages = [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": question}]

    while True:
        reply = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            reasoning_effort=EFFORT or omit,
        )
        message = reply.choices[0].message
        if not message.tool_calls:
            return message.content

        messages.append(message)
        for call in message.tool_calls:
            arguments = json.loads(call.function.arguments)
            result = FUNCTIONS[call.function.name](**arguments)
            print("  lookup:", call.function.name, arguments)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

The first run goes straight to OpenAI: a plain `OpenAI()` client and the model's name from `models.env`.
`timeout=60` gives up on a request after 60 seconds, so a request that gets stuck cannot freeze a cell.

In [7]:
HELENA = "How many invoices does Helena Holý have, what do they add up to, and who is her support rep?"

direct_client = OpenAI(timeout=60)

print(ask_desk(direct_client, MODEL, HELENA))

  lookup: find_customer {'name': 'Helena Holý'}


  lookup: get_invoices {'customer_id': 6}
  lookup: get_support_rep {'customer_id': 6}


Helena Holý has 7 invoices totaling $49.62, and her support rep is Steve Johnson.


Three lookups, then the answer: 7 invoices, \$49.62, Steve Johnson. They agree with the table above. `find_customer` ran first,
because the other two need the `customer_id` it returns.

We have a working agent. Nothing in it uses LiteLLM.

**Checkpoint A.** The lookups by hand give Helena 7 invoices, \$49.62 and Steve Johnson. The agent
prints three `lookup:` lines, `find_customer` first, and answers with the same three facts.

---

## Part B: Describe a primary and a backup deployment in the gateway's config

The agent calls one model at one address. If that address stops answering, so does the desk. A LiteLLM gateway
between the agent and the model can hold a second deployment to fall back on. We write its configuration file
first.

![The desk agent calls the LiteLLM gateway at one address, with the gateway's key and the name desk. Inside the gateway, the deployment desk points at the primary model and the deployment desk-backup at the backup model, with a dotted fallback arrow from desk to desk-backup. The reply goes back to the agent with headers that name the deployment, the fallbacks and the cost.](resources/lab21_gateway.png)

Light green is our own code, the dark boxes are models, white and grey are LiteLLM.

The gateway reads a YAML file. We write it from this notebook, so that the model names still come from
`models.env`, and print it. The file goes in `outputs/`, a folder beside this notebook that git ignores.

In [8]:
OUTPUTS = Path("outputs")
OUTPUTS.mkdir(exist_ok=True)
CONFIG_FILE = OUTPUTS / "gateway_config.yaml"

OPENAI_ADDRESS = "https://api.openai.com/v1"

config = f"""model_list:
  - model_name: desk
    litellm_params:
      model: openai/{MODEL}
      api_key: os.environ/OPENAI_API_KEY
      api_base: {OPENAI_ADDRESS}
    model_info:
      id: desk-primary
  - model_name: desk-backup
    litellm_params:
      model: openai/{settings["MODEL_STRONG"]}
      api_key: os.environ/OPENAI_API_KEY
    model_info:
      id: desk-backup-1

router_settings:
  fallbacks:
    - desk: [desk-backup]

general_settings:
  master_key: lab-21-local-key
"""

CONFIG_FILE.write_text(config)
print(config)

model_list:
  - model_name: desk
    litellm_params:
      model: openai/gpt-6-luna
      api_key: os.environ/OPENAI_API_KEY
      api_base: https://api.openai.com/v1
    model_info:
      id: desk-primary
  - model_name: desk-backup
    litellm_params:
      model: openai/gpt-6-luna
      api_key: os.environ/OPENAI_API_KEY
    model_info:
      id: desk-backup-1

router_settings:
  fallbacks:
    - desk: [desk-backup]

general_settings:
  master_key: lab-21-local-key



Read the file block by block:

- **`model_list`** holds two deployments. The agent will ask for the name `desk`; its deployment has the id
  `desk-primary`. The second name, `desk-backup`, is there for when the first fails; its deployment has the id
  `desk-backup-1`. Each `litellm_params` says what the gateway really calls: the vendor and the model
  (`openai/<model>`). The fallback line below refers to names (`desk`, `desk-backup`), and the headers we read
  later print ids (`desk-primary`, `desk-backup-1`).
- **`api_key: os.environ/OPENAI_API_KEY`** tells the gateway to read the vendor's key from its own environment
  variable of that name. The key itself is not in the file.
- **`api_base`** is the address the gateway sends `desk`'s calls to. We write OpenAI's usual address out, because
  Part E changes this one line.
- **`router_settings`** holds the fallback: when `desk` fails, try `desk-backup`.
- **`master_key`** is the key of the gateway itself. A program that calls the gateway sends this key, and never
  the vendor's. We made this one up for a gateway that runs only on this machine. A real gateway gets a long
  random key from an environment variable.

Both deployments here are OpenAI models, reached with one key, and `models.env` may name the same model for both. That is enough to see a fallback work. Going further says how to put the backup at
another vendor.

### Start the gateway from the notebook

Installing LiteLLM also installed a command, `litellm`. In a terminal, in this lab's folder, we would type
`uv run litellm --config outputs/gateway_config.yaml --host 127.0.0.1 --port 4000` and leave it running. We start and stop the
gateway several times in this notebook, so we put the steps in two small functions.

The cell first finds a free port. It starts at 4000, LiteLLM's usual port. `connect_ex` returns 0 when some
program already answers at a port; the loop then says so and tries the next number. `PORT` and `GATEWAY` hold
the result, and every later cell uses them.

`start_gateway()` has three numbered steps:

- Step 1 checks that our port is still free. It is not when a gateway we started is still running; the message
  says how to stop it.
- Step 2 starts the gateway. `subprocess.Popen` starts a command and lets the notebook carry on while it runs. The command is this kernel's
  Python (`sys.executable`), then `-m litellm.proxy.proxy_cli`, which is what the `litellm` command runs, then
  the config file, the host and the port. `--host 127.0.0.1` makes the gateway answer only programs on this
  machine; without it the gateway listens on every network the machine is connected to. Everything the gateway
  prints goes to `outputs/gateway.log`.
- Step 3 waits. The gateway needs about ten seconds to start. The loop asks it for its list of models once a
  second, with an OpenAI client pointed at it. `max_retries=0` stops the OpenAI client from trying again by itself
  (by default it retries a failed request twice), so each pass of the loop is one attempt. While the gateway is still starting, the client raises `APIConnectionError` and we
  wait. When it answers, we return the process. (`process.poll()` is `None` as long as the process is alive.)

`stop_gateway(process)` asks the process to stop and waits until it has, which frees the port.

If the cell prints a port other than 4000, use that number in the terminal command above.

In [9]:
import socket
import subprocess
import sys
import time

from openai import APIConnectionError

# The first free port, starting at LiteLLM's usual 4000
PORT = 4000
while True:
    with socket.socket() as probe:
        if probe.connect_ex(("127.0.0.1", PORT)) != 0:
            break
    print("port", PORT, "is in use, trying the next one")
    PORT += 1

GATEWAY = f"http://127.0.0.1:{PORT}"
GATEWAY_KEY = "lab-21-local-key"
print("the gateway will use port", PORT)


def start_gateway():
    """Start the LiteLLM gateway with our config file, and return its process once it answers."""
    # 1. Is our port still free? It is not when a gateway we started is still running
    with socket.socket() as probe:
        port_in_use = probe.connect_ex(("127.0.0.1", PORT)) == 0
    assert not port_in_use, f"Port {PORT} is in use. Stop the running gateway first: stop_gateway(gateway)."

    # 2. Start the gateway as a second program; what it prints goes to outputs/gateway.log
    with open(OUTPUTS / "gateway.log", "w") as log:
        process = subprocess.Popen(
            [sys.executable, "-m", "litellm.proxy.proxy_cli", "--config", str(CONFIG_FILE), "--host", "127.0.0.1", "--port", str(PORT)],
            stdout=log,
            stderr=subprocess.STDOUT,
        )

    # 3. Ask for its model list once a second until it answers
    checker = OpenAI(base_url=GATEWAY + "/v1", api_key=GATEWAY_KEY, max_retries=0)
    while True:
        assert process.poll() is None, "The gateway stopped while starting. Read outputs/gateway.log."
        try:
            checker.models.list()
            return process
        except APIConnectionError:
            time.sleep(1)


def stop_gateway(process):
    """Ask the gateway to stop, and wait until it has."""
    process.terminate()
    process.wait()

the gateway will use port 4000


In [10]:
gateway = start_gateway()

gateway_client = OpenAI(base_url=GATEWAY + "/v1", api_key=GATEWAY_KEY, timeout=60)

print("the gateway is up and offers:", [model.id for model in gateway_client.models.list()])

the gateway is up and offers: ['desk', 'desk-backup']


The gateway is up. `gateway_client` is an ordinary `OpenAI` client with two things changed: `base_url` is the
gateway's address, and `api_key` is the gateway's key. Asked which models exist, the gateway answers with our two
names. The vendor's model name does not appear.

Ours took port 4000, which was free. When 4000 is busy (a LiteLLM gateway running in Docker usually listens there), the cell prints
`port 4000 is in use, trying the next one` and tries 4001.

**Checkpoint B.** The config file prints with `desk`, `desk-backup`, the fallback and the gateway's key. The
gateway offers `['desk', 'desk-backup']`.

---

## Part C: Point the desk agent at the gateway by changing the client and the model's name

Here is the call from Part A again, and under it the call we are about to make:

```python
ask_desk(direct_client,  MODEL,  HELENA)    # Part A: straight to OpenAI
ask_desk(gateway_client, "desk", HELENA)    # now: through the gateway
```

Two arguments differ. The client is the one pointed at the gateway, and the model is the name `desk` from the
config file. `ask_desk` itself, its tools and its instructions are untouched.

In [11]:
print(ask_desk(gateway_client, "desk", HELENA))

  lookup: find_customer {'name': 'Helena Holý'}


  lookup: get_invoices {'customer_id': 6}
  lookup: get_support_rep {'customer_id': 6}


Helena Holý has 7 invoices totaling $49.62, and her support rep is Steve Johnson.


The same three lookups and the same three facts: 7 invoices, \$49.62 and Steve Johnson.

**Key lesson:** the gateway answers in the shape of OpenAI's API, so anything that can call OpenAI can call the
gateway. The agent did not change, and nothing in it names the vendor or the model behind `desk`. Those
names are now in the gateway's config file, and the vendor's key stays with the gateway.

**Checkpoint C.** Through the gateway the agent prints the same three lookups and answers with Helena's
7 invoices, \$49.62 and Steve Johnson.

---

## Part D: Read which deployment answered, and what it cost, from the gateway's headers

The answer in Part C looked exactly like the one in Part A. That is the point of a gateway, and it leaves a
question open: how do we know what happened behind it?

The gateway tells us in the reply's **headers**: lines of `name: value` that travel with every HTTP reply, beside
its body. First we look at them on one small request. The OpenAI client shows a reply's headers when we call
through `with_raw_response`. The cost header is a number in scientific notation, so the last line also prints it
as dollars with eight decimals.

In [12]:
raw = gateway_client.chat.completions.with_raw_response.create(
    model="desk",
    messages=[{"role": "user", "content": "Reply with the one word: ready"}],
    reasoning_effort=EFFORT or omit,
)

for header in ["x-litellm-model-group", "x-litellm-model-id", "x-litellm-attempted-fallbacks", "x-litellm-response-cost"]:
    print(header + ":", raw.headers.get(header))
print("the cost in dollars:", f"${float(raw.headers['x-litellm-response-cost']):.8f}")

x-litellm-model-group: desk
x-litellm-model-id: desk-primary
x-litellm-attempted-fallbacks: 0
x-litellm-response-cost: 3.2999999999999997e-06
the cost in dollars: $0.00000330


Four of the headers the gateway adds to a reply:

| Header | What it says | In our run |
|---|---|---|
| `x-litellm-model-group` | the name that answered | `desk` |
| `x-litellm-model-id` | the deployment that answered | `desk-primary` |
| `x-litellm-attempted-fallbacks` | how many fallbacks the gateway tried | `0` |
| `x-litellm-response-cost` | the gateway's estimate of the cost, in US dollars | `3.2999999999999997e-06`, which is \$0.00000330 |

The cost is an estimate: the reply's token counts times the list prices in a table that ships with LiteLLM. The
bill comes from the vendor.

### Keep the headers of every request the agent makes

The agent makes several model calls for one question, and we want the headers of each without touching
`ask_desk`. The OpenAI client can call a function of ours for every HTTP reply it receives. Such a function is
called a **hook**.

- `keep_headers(response)` is our hook. It adds the reply's headers to the list `gateway_headers`. The `response`
  it receives is the raw HTTP reply, not the `ChatCompletion`: it has `.headers` and no `.choices`.
- `logged_client` is `gateway_client` again, with one more argument: `http_client=DefaultHttpxClient(event_hooks=...)`
  hands the hook to the HTTP library inside the OpenAI client.

In [13]:
from openai import DefaultHttpxClient

gateway_headers = []


def keep_headers(response):
    """Called by the client for every HTTP reply: keep the reply's headers."""
    gateway_headers.append(response.headers)


logged_client = OpenAI(
    base_url=GATEWAY + "/v1",
    api_key=GATEWAY_KEY,
    timeout=60,
    http_client=DefaultHttpxClient(event_hooks={"response": [keep_headers]}),
)

One small request through `logged_client`, to see the hook work. We do not print the reply this time. We print
what the hook put in the list.

In [14]:
logged_client.chat.completions.create(
    model="desk",
    messages=[{"role": "user", "content": "Reply with the one word: ready"}],
    reasoning_effort=EFFORT or omit,
)

print("replies in the list:", len(gateway_headers))
print("x-litellm-model-id:", gateway_headers[0]["x-litellm-model-id"])
print("x-litellm-response-cost:", gateway_headers[0]["x-litellm-response-cost"])

replies in the list: 1
x-litellm-model-id: desk-primary
x-litellm-response-cost: 3.2999999999999997e-06


One reply is in the list, and its headers are the kind we read above: the deployment `desk-primary` and a cost.

The next function only prints what is in that list: one line per request, with the cost as dollars, and then
the total of the costs. We call it at once on the one request we have.

In [15]:
def show_gateway_headers():
    """Print one line per request the gateway answered, then the total of its cost estimates."""
    total = 0.0
    for number, headers in enumerate(gateway_headers, start=1):
        cost = float(headers["x-litellm-response-cost"])
        total += cost
        print(f"request {number}: answered by {headers['x-litellm-model-id']}"
              f" | fallbacks: {headers['x-litellm-attempted-fallbacks']} | cost: ${cost:.6f}")
    print(f"total cost: ${total:.6f}")


show_gateway_headers()

request 1: answered by desk-primary | fallbacks: 0 | cost: $0.000003
total cost: $0.000003


One request, answered by `desk-primary`, and a total equal to its cost. Now the agent's question, through `logged_client`. `gateway_headers.clear()` empties the list first, and we time the
whole question.

In [16]:
gateway_headers.clear()
start = time.time()

print(ask_desk(logged_client, "desk", HELENA))

healthy_seconds = round(time.time() - start, 1)
print()
show_gateway_headers()
print("seconds:", healthy_seconds)

  lookup: find_customer {'name': 'Helena Holý'}


  lookup: get_invoices {'customer_id': 6}
  lookup: get_support_rep {'customer_id': 6}


Helena Holý has 7 invoices totaling $49.62. Her support rep is Steve Johnson.

request 1: answered by desk-primary | fallbacks: 0 | cost: $0.000036
request 2: answered by desk-primary | fallbacks: 0 | cost: $0.000057
request 3: answered by desk-primary | fallbacks: 0 | cost: $0.000054
total cost: $0.000147
seconds: 8.2


Under the answer, what the gateway's headers show:

- **Three requests,** one per model call of the loop: the first asked for `find_customer`, the second for the other two lookups together, and the third wrote the answer.
- **Each was answered by `desk-primary`,** with 0 fallbacks.
- **The question cost about \$0.000147** by the gateway's estimate. The second and third requests cost more than
  the first, because each carries the whole conversation so far, with the earlier tool calls and results. The
  second cost a little more than the third although the third carried more. In its reply to the second request
  the model wrote two tool calls, about twice as many output tokens as the one-sentence answer, and an output token costs five times as much as an input token (0.50 against 0.10
  dollars per million in `models.env`).
- **It took 8.2 seconds.**

A team that runs the gateway with a database gets the same facts for every request of every program, on the
gateway's admin page (Going further). The headers give them to us with nothing but the reply.

**Checkpoint D.** The single request prints the name `desk`, the deployment `desk-primary`, 0 fallbacks and a
cost. The request through `logged_client` leaves one reply in the list. The agent's question shows as three requests, all answered by `desk-primary`, with a total cost of
a fraction of a cent.

---

## Part E: Break the primary on purpose, and compare the agent with and without the gateway

Now the failure. We cannot switch OpenAI off, so we imitate it with an address where nothing answers:
`http://127.0.0.1:9/v1`, a port on this machine where no program listens, so every request is refused at once.
We call it the dead address.

### The agent wired straight to the dead address

First the agent of Part A, with its client pointed at that address. `max_retries=0` makes the client give up
after one attempt. The call is wrapped in `try` and `except`, so that the error prints as one line.

In [17]:
NOWHERE = "http://127.0.0.1:9/v1"

dead_client = OpenAI(base_url=NOWHERE, timeout=60, max_retries=0)

try:
    print(ask_desk(dead_client, MODEL, HELENA))
except Exception as error:
    print(type(error).__name__, "|", error)

APIConnectionError | Connection error.


`APIConnectionError | Connection error.`

No lookup ran and the rep has no answer. An agent that calls one address has no second address to try.

### The same failure behind the gateway

⚠️ We break the primary on purpose. We stop the gateway, write the config file again with one line changed (the
`api_base` of `desk` now points at the dead address) and start the gateway again. `desk-backup` is unchanged.
This is a test of the fallback, and the broken address is its only purpose.

`config.replace(OPENAI_ADDRESS, NOWHERE)` returns the config text with that one address swapped. We print the
changed line to check.

In [18]:
stop_gateway(gateway)

broken_config = config.replace(OPENAI_ADDRESS, NOWHERE)
CONFIG_FILE.write_text(broken_config)
print([line.strip() for line in broken_config.splitlines() if "api_base" in line])

gateway = start_gateway()
print("the gateway is up again and offers:", [model.id for model in gateway_client.models.list()])

['api_base: http://127.0.0.1:9/v1']


the gateway is up again and offers: ['desk', 'desk-backup']


The gateway is running again with `desk` pointed at the dead address. It still offers both names: it does not
test its deployments when it starts.

Now the line from Part D again, unchanged. Before you run it: will the rep get an answer, and which deployment
will the three requests show?

In [19]:
gateway_headers.clear()
start = time.time()

print(ask_desk(logged_client, "desk", HELENA))

broken_seconds = round(time.time() - start, 1)
print()
show_gateway_headers()
print("seconds:", broken_seconds, "| with the primary working:", healthy_seconds)

  lookup: find_customer {'name': 'Helena Holý'}


  lookup: get_invoices {'customer_id': 6}
  lookup: get_support_rep {'customer_id': 6}


Helena Holý has 7 invoices totaling $49.62. Her support rep is Steve Johnson.

request 1: answered by desk-backup-1 | fallbacks: 1 | cost: $0.000036
request 2: answered by desk-backup-1 | fallbacks: 1 | cost: $0.000057
request 3: answered by desk-backup-1 | fallbacks: 1 | cost: $0.000054
total cost: $0.000147
seconds: 18.1 | with the primary working: 8.2


The rep gets the answer: 7 invoices, \$49.62, Steve Johnson. The agent's code is the code of Part D, and it saw no error.

The headers show what the gateway did:

- **Every request was answered by `desk-backup-1`,** the id of the backup deployment, after 1 fallback each. The agent asked for `desk` all
  three times.
- **The cost did not change (\$0.000147 both times),** because here the backup is a model with the same
  prices. With a more expensive backup, an outage of the primary would show up here as a higher bill.
- **The question took 18.1 seconds, against 8.2.** For each request the gateway first tried
  `desk-primary`, and by default it retries a failed call twice before it falls back. It did that again for every
  request, three times in a row. The rep got the facts, later.

**Important:** a fallback turns an outage into a slower answer, and nobody is told. The agent cannot see it, and
the rep only waits longer. The headers are where the outage shows, so someone, or some program, has to read
them. How long the gateway keeps trying a dead deployment is a setting too; Going further names it.

**The limit of this test:** both deployments are OpenAI models reached with one key. That protects the desk when
one deployment fails. If the vendor itself is down, both fail, and the backup has to be at another vendor.

### Stop the gateway

In [20]:
stop_gateway(gateway)
print("gateway stopped")

gateway stopped


⚠️ Always run this cell before you close the notebook or restart its kernel. A gateway that is left running keeps
its port, and the next run of this notebook has to take the next one (see Troubleshooting).

**Checkpoint E.** Wired straight to the dead address, the agent prints a connection error and no lookups. Behind
the gateway it answers with Helena's 7 invoices, \$49.62 and Steve Johnson; every request shows
`desk-backup-1` and 1 fallback; and the question takes longer than in Part D (ours: 18.1
seconds against 8.2). The last cell prints `gateway stopped`.

---

## What you can do now

- Build a desk agent from three lookups, their tool schemas and a loop on Chat Completions (Part A).
- Write a gateway config with a primary deployment, a backup and a fallback, and start and stop the gateway from
  a notebook (Part B).
- Point an existing agent at the gateway by changing its client and its model name (Part C).
- Read from the reply headers which deployment answered each request and what the gateway estimates it cost
  (Part D).
- Test a fallback by breaking the primary on purpose, and say what the fallback saved, what it cost in time and
  what it hides (Part E).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `port 4000 is in use, trying the next one` | another program is using that port: a LiteLLM gateway in Docker, or a gateway of this lab that was never stopped (the kernel was restarted before the stop cell ran) | Nothing to fix: the cell takes the next free port. To stop a leftover gateway of this lab, in a terminal: `pkill -f litellm.proxy.proxy_cli` on Linux or a Mac; on Windows end the `python` process that runs it in the Task Manager |
| `AssertionError: Port 4000 is in use. Stop the running gateway first: stop_gateway(gateway).` (or with another port number) | `start_gateway()` was called while the gateway it started earlier is still running | run `stop_gateway(gateway)`, then start again |
| `AssertionError: The gateway stopped while starting. Read outputs/gateway.log.` | the gateway could not start; the last lines of the log say why | open `outputs/gateway.log`; a missing package means the environment needs `uv sync` |
| `Error code: 400 - {'error': {'message': "/chat/completions: Invalid model name passed in model=...` | the agent asked the gateway for a name it does not offer, for example the vendor's own model name | pass `"desk"`; `gateway_client.models.list()` shows the names |
| `APIConnectionError: Connection error.` from `gateway_client` | the gateway is not running | run `gateway = start_gateway()` again |
| `InternalServerError: Error code: 500 - litellm.InternalServerError: InternalServerError: OpenAIException - Connection error.` followed by `Fallback to desk-backup also failed` | every deployment in the fallback list failed | expected in the Challenge; otherwise check the `api_base` lines in `outputs/gateway_config.yaml` |
| `sqlite3.OperationalError: unable to open database file` | `data/chinook.db` is missing from the repository folder | restore the file (clone the repository again, or `git checkout data/chinook.db`) |

## Challenge

1. **Give the desk a second job with its own name.** The desk also wants a five-word summary of each question for
   its ticket log. Start from the text in `config` (Part B's working file), add a third deployment with the name
   `notes` and the id `notes-primary`, write it to `CONFIG_FILE`, call `start_gateway()` and ask `notes` to summarise Helena's question. Read the headers of that request. Expected: the
   gateway offers three names, a summary of about five words, and `x-litellm-model-id: notes-primary` with 0
   fallbacks.
2. **Break the backup too.** Point both `desk` and `desk-backup` at the dead address (`desk-backup` has no
   `api_base` line yet, so give it one), restart the gateway and
   ask Helena's question through `logged_client`, inside `try` and `except`. Expected: no lookups, and an error
   from the gateway that says the fallback failed as well, after a wait of about half a minute (ours: 29.2 seconds), because the OpenAI client sends the failed request twice more and the gateway runs its retries
each time. Then say what the rep
   should see in that case, and where in `ask_desk` you would put it.

Stop the gateway when you are done. The worked solution, run with its outputs, is in
`solved/21b_litellm_at_the_desk_solution.ipynb`.

## Going further

- **Put the backup at another vendor.** With a second key in the gateway's environment, change `desk-backup` to
  `model: gemini/<model>` with `api_key: os.environ/GEMINI_API_KEY`. The agent does not change. Part I of
  notebook 1 does the same with a `Router`.
- **A request that is wrong fails on the backup too.** A fallback helps when a deployment fails. It cannot repair
  a bad request. When we sent a `Router` with a fallback a request with `temperature=9`, the first deployment
  refused it with a 400 error, the Router tried the fallback, and the fallback refused it for the same reason.
- **Retries and cooldowns.** In Part E the gateway tried the dead primary again for every request.
  `num_retries` in `router_settings` sets how often a failed call is retried before the fallback, and
  `allowed_fails` with `cooldown_time` take a deployment that keeps failing out of use for a while. The
  [fallbacks page](https://docs.litellm.ai/docs/proxy/reliability) describes them.
- **The admin page, keys and budgets (needs Docker).** A gateway with a database behind it has an admin page
  at `/ui` (pictured in notebook 1). There a team issues **virtual keys**: a key per program or per person, each
  with the models it may use and a budget in dollars, after which the gateway refuses its requests. The page also
  lists every request with its deployment and its cost, which is what we read from the headers in Part D. The
  [Docker quick start](https://docs.litellm.ai/docs/proxy/docker_quick_start) starts a gateway with a database,
  and [Virtual keys](https://docs.litellm.ai/docs/proxy/virtual_keys) shows how to create a key with
  `max_budget`. This lab does not run it, so that nothing here needs Docker.
- **Other clients.** Anything that speaks OpenAI's API can use the gateway: the OpenAI client in other languages,
  and agent frameworks that accept a `base_url`. The page
  [Using the gateway from other tools](https://docs.litellm.ai/docs/proxy/user_keys) has examples.

## Resources

- [LiteLLM gateway: quick start](https://docs.litellm.ai/docs/proxy/quick_start): starting the gateway from the
  command line
- [The config file](https://docs.litellm.ai/docs/proxy/configs): `model_list`, `router_settings`,
  `general_settings` and `os.environ/`
- [Fallbacks](https://docs.litellm.ai/docs/proxy/reliability): fallbacks, retries and cooldowns
- [Response headers](https://docs.litellm.ai/docs/proxy/response_headers): every `x-litellm-` header
- [Spend tracking](https://docs.litellm.ai/docs/proxy/cost_tracking): how the gateway records cost when it has a
  database
- [Virtual keys](https://docs.litellm.ai/docs/proxy/virtual_keys) and
  [budgets](https://docs.litellm.ai/docs/proxy/users): keys per program with a spending limit
- [Router](https://docs.litellm.ai/docs/routing): the same deployments and fallbacks inside one Python program
- [OpenAI: function calling](https://developers.openai.com/api/docs/guides/function-calling): the tool schemas
  and the tool round of Part A

Back to [notebook 1](21a_meet_litellm.ipynb). Next:
[Lab 22: How to re-engineer a process with agents](../22_Process_Reengineering/README.md). Back to
[the lab list](../README.md).